# Minimal RAG Solution

## Overview
This notebook demonstrates a minimal Retrieval-Augmented Generation (RAG) system running locally, using modular components.
It uses:
- **Source**: Wikipedia articles
- **Embeddings**: HuggingFace (all-MiniLM-L6-v2)
- **Vector Store**: ChromaDB (local persistence)
- **LLM**: Ollama (Llama 3.2 or Phi-3)
- **Orchestration**: LangChain

## Setup
To run this notebook, you need to install the dependencies. You can do this by running the following command in your terminal or in a code cell:

```bash
pip install -r requirements.txt
```

In [7]:
import os
import warnings

# Suppress warnings for a cleaner demo output
warnings.filterwarnings('ignore')

from langchain_community.document_loaders import WikipediaLoader

# Import modular components
from src.splitter import split_documents
from src.vectorstore import get_vectorstore_retriever
from src.rag import get_rag_chain

## 1. Data Ingestion
We will load content from Wikipedia. For this demo, we'll focus on AI-related topics to build our knowledge base.

In [8]:
# Define topics to search on Wikipedia
topics = ["Retrieval-augmented generation", "Generative artificial intelligence", "Large language model"]

docs = []
for topic in topics:
    loader = WikipediaLoader(query=topic, load_max_docs=2)
    docs.extend(loader.load())

print(f"Loaded {len(docs)} documents total.")
# Preview the first doc metadata
print(f"Sample source: {docs[0].metadata['source']}")

Loaded 6 documents total.
Sample source: https://en.wikipedia.org/wiki/Retrieval-augmented_generation


In [17]:
# print(docs[1].page_content)

## 2. Text Splitting (Semantic Layer Preparation)
We need to break these large documents into smaller chunks that fit into the embedding model's context window.

In [18]:
splits = split_documents(docs=docs)
print(f"Split {len(docs)} documents into {len(splits)} chunks.")

Split 6 documents into 34 chunks.


## 3. Vector Store & Embeddings
We use `all-MiniLM-L6-v2` for embeddings as it's efficient for local CPU execution. ChromaDB is used as the vector store.

In [19]:
retriever = get_vectorstore_retriever(
    documents=splits,
    persist_directory="./chroma_db"
)

W0128 22:34:38.437000 29228 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


: 

## 4. RAG Pipeline Setup
We configure the LLM (Ollama running llama3.2) and the prompt template.

In [ ]:
# Ensure you have run `ollama pull llama3.2` or `ollama pull phi3` in your terminal
rag_chain = get_rag_chain(retriever=retriever, model_name="llama3.2")

## 5. Execution
Let's test the system with a query.

In [ ]:
query = "What is Retrieval-augmented generation?"

print(f"Processing query: {query}...")
response = rag_chain.invoke(query)

print("\nResponse:")
print(response)

In [ ]:
# Check what sources were retrieved
retrieved_docs = retriever.invoke(query)
print("\nSources Used:")
for i, doc in enumerate(retrieved_docs):
    print(f"{i+1}. {doc.metadata.get('title', 'Unknown')} - Source: {doc.metadata.get('source', 'N/A')}")